In [0]:
%pip install scikit-learn matplotlib seaborn

In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, IntegerType, TimestampType
)
from delta.tables import DeltaTable
from mlflow.models.signature import infer_signature
from datetime import date

In [0]:
dbutils.widgets.text("job_parameters", "{}")

parameters = json.loads(dbutils.widgets.get("job_parameters"))

print(f"Raw widget value : {repr(dbutils.widgets.get('job_parameters'))}")
print(f"Parsed keys      : {list(parameters.keys())}")

# ── Identity & routing ────────────────────────────────────────────────────
catalog          = parameters.get("catalog")
_source_view     = parameters.get("source_view")
_target_table    = parameters.get("target_table")
primary_keys     = parameters.get("primary_keys")
job_id           = parameters.get("job_id")
parent_job       = parameters.get("parent_job_id", job_id)
partition        = parameters.get("partition")
default_value_flag = parameters.get("default_value_flag", True)

# ── ML params ─────────────────────────────────────────────────────────────
feature_cols     = parameters.get("feature_cols")
model_type       = parameters.get("model_type", "kmeans")
experiment_path  = parameters.get("experiment_path")
random_state     = int(parameters.get("random_state", 42))
categorical_cols = parameters.get("categorical_cols", [])
fill_with_zero   = parameters.get("fill_with_zero", [])

# ── K-Means specific params ───────────────────────────────────────────────
# Range of k values to evaluate via Elbow + Silhouette
k_min            = int(parameters.get("k_min", 3))
k_max            = int(parameters.get("k_max", 10))
# Final k to use for production model (set after reviewing elbow plot)
# If None — auto-selected as best silhouette score
k_final          = parameters.get("k_final", None)
if k_final is not None:
    k_final = int(k_final)

# KMeans hyperparams
kmeans_params    = parameters.get("kmeans_params", {
    "init"      : "k-means++",
    "n_init"    : 10,
    "max_iter"  : 300,
    "tol"       : 1e-4,
    "random_state": random_state,
})

# ── Segment name mapping ──────────────────────────────────────────────────
# Maps cluster_id (0, 1, 2...) → business segment name
# Populated after reviewing cluster profiles
# If empty — auto-labelled as "Segment_0", "Segment_1" etc.
segment_names    = parameters.get("segment_names", {})

# ── Passthrough cols for output ───────────────────────────────────────────
passthrough_cols = parameters.get("passthrough_cols",
                                  ["customer_state", "customer_city"])

# ── Validate required params ──────────────────────────────────────────────
required = {
    "catalog"        : catalog,
    "source_view"    : _source_view,
    "target_table"   : _target_table,
    "feature_cols"   : feature_cols,
    "experiment_path": experiment_path,
    "primary_keys"   : primary_keys,
    "job_id"         : job_id,
    "partition"      : partition,
}
missing = [k for k, v in required.items() if not v]
if missing:
    raise ValueError(f"❌ Missing required parameters: {missing}")

# ── Construct table names ────────────────────────────────────────────────
source_view    = f"{catalog}.{_source_view}"
target_table   = f"{catalog}.{_target_table}"
registry_table = f"{catalog}.gold.model_registry"

print("=" * 60)
print("  CUSTOMER SEGMENTATION — PARAMETERS LOADED")
print("=" * 60)
print(f"  Source view    : {source_view}")
print(f"  Target table   : {target_table}")
print(f"  Feature count  : {len(feature_cols)}")
print(f"  K range        : {k_min} → {k_max}")
print(f"  K final        : {'auto (best silhouette)' if k_final is None else k_final}")
print(f"  Experiment     : {experiment_path}")
print(f"  Job ID         : {job_id}")
print(f"  Partition      : {partition}")
print("=" * 60)

In [0]:
experiment = mlflow.get_experiment_by_name(experiment_path)
if experiment is None:
    mlflow.create_experiment(experiment_path)
    print(f"✅ Created new experiment : {experiment_path}")
else:
    print(f"✅ Reusing experiment     : {experiment_path}")
    print(f"   Experiment ID         : {experiment.experiment_id}")

mlflow.set_experiment(experiment_path)
print(f"✅ MLflow tracking active")

In [0]:
df = spark.table(source_view).toPandas()

print(f"✅ Loaded {source_view}")
print(f"   Rows     : {df.shape[0]:,}")
print(f"   Columns  : {df.shape[1]}")

all_expected = feature_cols + primary_keys + passthrough_cols
missing_cols = [c for c in all_expected if c not in df.columns]
if missing_cols:
    raise ValueError(f"❌ Missing columns: {missing_cols}")

print(f"✅ All expected columns present")
print(f"\nFeature summary:")
print(df[feature_cols].describe().round(3).to_string())

In [0]:
# ── Encode categoricals ───────────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder

encoders             = {}
encoded_feature_cols = feature_cols.copy()

for col in categorical_cols:
    if col in df.columns:
        le          = LabelEncoder()
        encoded_col = f"{col}_encoded"
        df[encoded_col] = le.fit_transform(df[col].astype(str))
        encoders[col]   = le
        if col in encoded_feature_cols:
            encoded_feature_cols.remove(col)
        encoded_feature_cols.append(encoded_col)
        print(f"✅ Encoded {col} → {encoded_col}")

# ── Fill nulls ────────────────────────────────────────────────────────────
fill_with_median = [c for c in encoded_feature_cols if c not in fill_with_zero]
zero_cols        = [c for c in fill_with_zero   if c in encoded_feature_cols]
median_cols      = [c for c in fill_with_median if c in encoded_feature_cols]

df[zero_cols]   = df[zero_cols].fillna(0)
df[median_cols] = df[median_cols].fillna(df[median_cols].median(numeric_only=True))

# ── Cast to float ─────────────────────────────────────────────────────────
X_raw = df[encoded_feature_cols].astype(float)

remaining_nulls = X_raw.isna().sum().sum()
if remaining_nulls > 0:
    raise ValueError(f"❌ {remaining_nulls} nulls remain after filling")

# ── Keep primary keys + passthrough for output ───────────────────────────
df_keys = df[primary_keys + passthrough_cols].copy()

print(f"\n{'='*55}")
print(f"  PREPROCESSING COMPLETE")
print(f"{'='*55}")
print(f"  Feature matrix : {X_raw.shape}")
print(f"  Null values    : {remaining_nulls}")
print(f"{'='*55}")

In [0]:
# K-Means is distance-based — StandardScaler is mandatory
# Without scaling, high-magnitude features (monetary) dominate the clusters
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print(f"✅ StandardScaler applied")
print(f"   Input  mean range  : [{X_raw.mean().min():.2f}, {X_raw.mean().max():.2f}]")
print(f"   Scaled mean range  : [{X_scaled.mean(axis=0).min():.4f}, {X_scaled.mean(axis=0).max():.4f}]")
print(f"   Scaled std range   : [{X_scaled.std(axis=0).min():.4f}, {X_scaled.std(axis=0).max():.4f}]")

In [0]:
k_range      = range(k_min, k_max + 1)
inertias     = []
silhouettes  = []
db_scores    = []

print(f"Evaluating k = {k_min} to {k_max}...\n")
print(f"{'k':>4} {'Inertia':>14} {'Silhouette':>12} {'Davies-Bouldin':>16}")
print("-" * 50)

for k in k_range:
    km = KMeans(n_clusters=k, **kmeans_params)
    labels = km.fit_predict(X_scaled)

    inertia    = km.inertia_
    silhouette = silhouette_score(X_scaled, labels, sample_size=10000,
                                   random_state=random_state)
    db         = davies_bouldin_score(X_scaled, labels)

    inertias.append(inertia)
    silhouettes.append(silhouette)
    db_scores.append(db)

    print(f"{k:>4} {inertia:>14.1f} {silhouette:>12.4f} {db:>16.4f}")

# ── Elbow + Silhouette plot ───────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(k_range, inertias, "bo-", linewidth=2, markersize=8)
ax1.set_xlabel("Number of Clusters (k)")
ax1.set_ylabel("Inertia (Within-cluster SSE)")
ax1.set_title("Elbow Method — Inertia vs K")
ax1.grid(True, alpha=0.3)
for k, inertia in zip(k_range, inertias):
    ax1.annotate(str(k), (k, inertia), textcoords="offset points",
                 xytext=(0, 10), ha="center", fontsize=9)

ax2.plot(k_range, silhouettes, "rs-", linewidth=2, markersize=8)
ax2.set_xlabel("Number of Clusters (k)")
ax2.set_ylabel("Silhouette Score (higher = better)")
ax2.set_title("Silhouette Score vs K")
ax2.grid(True, alpha=0.3)
for k, s in zip(k_range, silhouettes):
    ax2.annotate(f"{s:.3f}", (k, s), textcoords="offset points",
                 xytext=(0, 10), ha="center", fontsize=9)

plt.suptitle("K-Means — Optimal K Selection", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("/tmp/elbow_silhouette.png", dpi=150, bbox_inches="tight")
plt.show()

# ── Auto-select best k ────────────────────────────────────────────────────
best_k_auto = list(k_range)[silhouettes.index(max(silhouettes))]
print(f"\n✅ Best k by silhouette score : {best_k_auto}  (score={max(silhouettes):.4f})")
print(f"   Review elbow plot above — override with k_final in job JSON if needed")

In [0]:
# ── Use k_final from params if provided, else auto-selected ───────────────
k_used = k_final if k_final is not None else best_k_auto

print(f"Training final K-Means with k={k_used}")
print(f"Source : {'job JSON (k_final)' if k_final is not None else 'auto (best silhouette)'}")

run_name = f"kmeans_k{k_used}_{job_id}"

with mlflow.start_run(run_name=run_name) as run:

    # ── Log params ────────────────────────────────────────────────────────
    mlflow.set_tags({
        "job_id"     : job_id,
        "parent_job" : parent_job,
        "partition"  : str(partition),
        "source_view": source_view,
        "model_type" : "kmeans",
        "run_type"   : "clustering"
    })

    mlflow.log_params({
        **kmeans_params,
        "k"          : k_used,
        "k_min"      : k_min,
        "k_max"      : k_max,
        "n_features" : len(encoded_feature_cols),
        "n_samples"  : len(X_scaled),
        "scaler"     : "StandardScaler",
        "source_view": source_view,
    })

    # ── Train pipeline: scaler + kmeans ──────────────────────────────────
    # Pipeline ensures scaler is saved with model for inference
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("kmeans", KMeans(n_clusters=k_used, **kmeans_params))
    ])
    pipeline.fit(X_raw)

    cluster_labels = pipeline.predict(X_raw)

    # ── Evaluation metrics ────────────────────────────────────────────────
    X_scaled_final = pipeline["scaler"].transform(X_raw)
    silhouette     = silhouette_score(X_scaled_final, cluster_labels,
                                       sample_size=10000,
                                       random_state=random_state)
    db_score       = davies_bouldin_score(X_scaled_final, cluster_labels)
    inertia        = pipeline["kmeans"].inertia_

    # Cluster size balance — penalise very uneven clusters
    cluster_sizes  = pd.Series(cluster_labels).value_counts().sort_index()
    size_std       = cluster_sizes.std()
    size_cv        = size_std / cluster_sizes.mean()  # coefficient of variation

    mlflow.log_metrics({
        "silhouette_score"    : round(silhouette, 4),
        "davies_bouldin_score": round(db_score,   4),
        "inertia"             : round(inertia,     2),
        "cluster_size_cv"     : round(size_cv,     4),
        "min_cluster_size"    : int(cluster_sizes.min()),
        "max_cluster_size"    : int(cluster_sizes.max()),
    })

    # ── Cluster size distribution ─────────────────────────────────────────
    print(f"\n{'='*55}")
    print(f"  FINAL MODEL METRICS (k={k_used})")
    print(f"{'='*55}")
    print(f"  Silhouette Score      : {silhouette:.4f}  (higher = better, max=1)")
    print(f"  Davies-Bouldin Score  : {db_score:.4f}  (lower = better)")
    print(f"  Inertia               : {inertia:.1f}")
    print(f"  Cluster Size CV       : {size_cv:.4f}  (lower = more balanced)")
    print(f"{'='*55}")
    print(f"\n  Cluster size distribution:")
    for cid, size in cluster_sizes.items():
        pct = size / len(cluster_labels) * 100
        bar = "█" * int(pct / 2)
        print(f"  Cluster {cid} : {size:>6,}  ({pct:>5.1f}%)  {bar}")

    # ── Log k search metrics ──────────────────────────────────────────────
    for k_val, sil, db, ine in zip(k_range, silhouettes, db_scores, inertias):
        mlflow.log_metrics({
            f"k{k_val}_silhouette"    : round(sil, 4),
            f"k{k_val}_davies_bouldin": round(db,  4),
            f"k{k_val}_inertia"       : round(ine, 2),
        })

    # ── Log elbow plot ────────────────────────────────────────────────────
    mlflow.log_artifact("/tmp/elbow_silhouette.png")

    # ── Log model ─────────────────────────────────────────────────────────
    signature     = infer_signature(X_raw, cluster_labels)
    input_example = X_raw.head(5)

    mlflow.sklearn.log_model(
        pipeline,
        "kmeans_pipeline",
        signature     = signature,
        input_example = input_example
    )
    best_run_id = run.info.run_id

print(f"\n✅ Model logged — run_id: {best_run_id}")
mlflow.end_run()

In [0]:
# ── Attach cluster labels to feature data ────────────────────────────────
df_profile         = X_raw.copy()
df_profile["cluster_id"] = cluster_labels

# ── Compute cluster centroids in original scale ───────────────────────────
centroids = df_profile.groupby("cluster_id")[encoded_feature_cols].mean().round(3)

print("Cluster Centroids (original scale):\n")
print(centroids.to_string())

# ── Heatmap of normalised centroids ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(max(14, len(encoded_feature_cols)), k_used + 1))
centroids_norm = (centroids - centroids.min()) / (centroids.max() - centroids.min() + 1e-9)
sns.heatmap(
    centroids_norm,
    annot=True, fmt=".2f", cmap="YlOrRd",
    linewidths=0.5, ax=ax,
    cbar_kws={"label": "Normalised Value (0=min, 1=max)"}
)
ax.set_title(f"Cluster Centroids Heatmap — K-Means k={k_used}", fontsize=14)
ax.set_xlabel("Features")
ax.set_ylabel("Cluster ID")
plt.tight_layout()
plt.savefig("/tmp/cluster_heatmap.png", dpi=150, bbox_inches="tight")
mlflow.log_artifact("/tmp/cluster_heatmap.png")
plt.show()

# ── PCA scatter plot ──────────────────────────────────────────────────────
pca        = PCA(n_components=2, random_state=random_state)
X_pca      = pca.fit_transform(X_scaled_final)
explained  = pca.explained_variance_ratio_

fig2, ax2  = plt.subplots(figsize=(10, 7))
colors     = plt.cm.Set1(np.linspace(0, 1, k_used))
for cid in range(k_used):
    mask = cluster_labels == cid
    label_name = segment_names.get(str(cid), f"Segment_{cid}")
    ax2.scatter(X_pca[mask, 0], X_pca[mask, 1],
                c=[colors[cid]], label=f"Cluster {cid}: {label_name}",
                alpha=0.4, s=10)

ax2.set_xlabel(f"PC1 ({explained[0]*100:.1f}% variance)")
ax2.set_ylabel(f"PC2 ({explained[1]*100:.1f}% variance)")
ax2.set_title(f"Customer Segments — PCA Projection (k={k_used})")
ax2.legend(loc="upper right", markerscale=3)
plt.tight_layout()
plt.savefig("/tmp/cluster_pca.png", dpi=150, bbox_inches="tight")
mlflow.log_artifact("/tmp/cluster_pca.png")
plt.show()

print(f"\n✅ PCA variance explained: PC1={explained[0]*100:.1f}% + PC2={explained[1]*100:.1f}% = {sum(explained)*100:.1f}% total")

In [0]:
# ── Assign segment name ───────────────────────────────────────────────────
# Uses segment_names from job JSON if provided
# Falls back to "Segment_0", "Segment_1"... if not

def get_segment_name(cluster_id):
    return segment_names.get(str(cluster_id), f"Segment_{cluster_id}")

# ── Compute centroid distance per customer ────────────────────────────────
# Distance to assigned cluster centroid — useful for identifying outliers
kmeans_model = pipeline["kmeans"]
distances    = kmeans_model.transform(X_scaled_final)  # shape: (n, k)
centroid_distances = np.array([
    distances[i, cluster_labels[i]]
    for i in range(len(cluster_labels))
])

# ── Build output DataFrame ────────────────────────────────────────────────
df_output = df_keys.copy()
df_output["cluster_id"]          = cluster_labels.astype(int)
df_output["segment_name"]        = [get_segment_name(c) for c in cluster_labels]
df_output["centroid_distance"]   = centroid_distances.round(4)
df_output["k_used"]              = k_used
df_output["silhouette_score"]    = round(silhouette, 4)
df_output["model_name"]          = f"{catalog}_customer_segments_kmeans"
df_output["model_version"]       = "1"
df_output["segmentation_date"]   = str(date.today())
df_output["job_id"]              = str(job_id)
df_output["partition"]           = str(partition)

print(f"✅ Output DataFrame built — {len(df_output):,} rows")
print(f"\nSegment distribution:")
seg_dist = df_output.groupby(["cluster_id", "segment_name"]).size().reset_index(name="count")
seg_dist["pct"] = (seg_dist["count"] / len(df_output) * 100).round(1)
print(seg_dist.to_string(index=False))

In [0]:
# ── Define output schema ──────────────────────────────────────────────────
output_schema = StructType([
    StructField("customer_unique_id",  StringType(),  nullable=False),
    StructField("customer_state",      StringType(),  nullable=True),
    StructField("customer_city",       StringType(),  nullable=True),
    StructField("cluster_id",          IntegerType(), nullable=False),
    StructField("segment_name",        StringType(),  nullable=False),
    StructField("centroid_distance",   DoubleType(),  nullable=True),
    StructField("k_used",              IntegerType(), nullable=False),
    StructField("silhouette_score",    DoubleType(),  nullable=True),
    StructField("model_name",          StringType(),  nullable=True),
    StructField("model_version",       StringType(),  nullable=True),
    StructField("segmentation_date",   StringType(),  nullable=False),
    StructField("job_id",              StringType(),  nullable=True),
    StructField("partition",           StringType(),  nullable=True),
])

output_cols = [f.name for f in output_schema]

predictions_spark = spark.createDataFrame(
    df_output[output_cols],
    schema=output_schema
)

# ── Create table if not exists ────────────────────────────────────────────
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {target_table} (
        customer_unique_id  STRING      NOT NULL,
        customer_state      STRING,
        customer_city       STRING,
        cluster_id          INT,
        segment_name        STRING,
        centroid_distance   DOUBLE,
        k_used              INT,
        silhouette_score    DOUBLE,
        model_name          STRING,
        model_version       STRING,
        segmentation_date   STRING,
        job_id              STRING,
        partition           STRING
    )
    USING DELTA
    COMMENT 'Customer segmentation labels — K-Means clustering pipeline'
    TBLPROPERTIES (
        'delta.autoOptimize.optimizeWrite' = 'true',
        'delta.autoOptimize.autoCompact'   = 'true'
    )
""")

print(f"✅ Target table ready : {target_table}")

# ── MERGE upsert ──────────────────────────────────────────────────────────
source_cols         = set(predictions_spark.columns)
target_cols         = set(spark.table(target_table).columns)
exclude_from_update = {"customer_unique_id"}

merge_set = {
    col: f"s.{col}"
    for col in source_cols
    if col not in exclude_from_update
    and col in target_cols
}

insert_set = {
    col: f"s.{col}"
    for col in source_cols
    if col in target_cols
}

DeltaTable.forName(spark, target_table) \
    .alias("t") \
    .merge(predictions_spark.alias("s"),
           "t.customer_unique_id = s.customer_unique_id") \
    .whenMatchedUpdate(set=merge_set) \
    .whenNotMatchedInsert(values=insert_set) \
    .execute()

print(f"✅ MERGE complete → {target_table}")

# ── Validate ──────────────────────────────────────────────────────────────
written       = spark.table(target_table)
written_count = written.count()

print(f"\n✅ Rows in table    : {written_count:,}")
print(f"✅ Segments present : {written.select('segment_name').distinct().count()}")
print(f"\n📋 Segment breakdown:")
display(
    written.groupBy("cluster_id", "segment_name")
    .agg(
        F.count("customer_unique_id").alias("customers"),
        F.round(F.avg("centroid_distance"), 4).alias("avg_centroid_dist")
    )
    .orderBy("cluster_id")
)

In [0]:
now = pd.Timestamp.now().to_pydatetime()

def safe_metric(col):
    val = mlflow.get_run(best_run_id).data.metrics.get(col, None)
    return float(val) if val is not None else None

registry_schema = StructType([
    StructField("model_name",       StringType(),    nullable=False),
    StructField("model_type",       StringType(),    nullable=True),
    StructField("target_col",       StringType(),    nullable=True),
    StructField("source_view",      StringType(),    nullable=True),
    StructField("run_id",           StringType(),    nullable=False),
    StructField("model_uri",        StringType(),    nullable=True),
    StructField("mlflow_version",   StringType(),    nullable=True),
    StructField("auc_roc",          DoubleType(),    nullable=True),
    StructField("f1_score",         DoubleType(),    nullable=True),
    StructField("precision_score",  DoubleType(),    nullable=True),
    StructField("recall_score",     DoubleType(),    nullable=True),
    StructField("cv_auc_mean",      DoubleType(),    nullable=True),
    StructField("cv_auc_std",       DoubleType(),    nullable=True),
    StructField("best_threshold",   DoubleType(),    nullable=True),
    StructField("score_col",        StringType(),    nullable=True),
    StructField("label_col",        StringType(),    nullable=True),
    StructField("feature_cols",     StringType(),    nullable=True),
    StructField("primary_keys",     StringType(),    nullable=True),
    StructField("job_id",           StringType(),    nullable=True),
    StructField("parent_job_id",    StringType(),    nullable=True),
    StructField("partition",        StringType(),    nullable=True),
    StructField("status",           StringType(),    nullable=True),
    StructField("registered_at",    TimestampType(), nullable=True),
    StructField("updated_at",       TimestampType(), nullable=True),
    StructField("retired_at",       TimestampType(), nullable=True),
    StructField("registered_by",    StringType(),    nullable=True),
])

model_name_reg = f"{catalog}_customer_segments_kmeans"
model_uri_reg  = f"runs:/{best_run_id}/kmeans_pipeline"

# ── Register in MLflow ────────────────────────────────────────────────────
try:
    model_details = mlflow.register_model(model_uri_reg, model_name_reg)
    mlflow_version = str(model_details.version)
    print(f"✅ Model registered : {model_name_reg} v{mlflow_version}")
except Exception as e:
    mlflow_version = None
    print(f"⚠️  Registration skipped: {str(e)}")

registry_row = [(
    model_name_reg,
    "kmeans",
    "cluster_id",
    source_view,
    str(best_run_id),
    model_uri_reg,
    mlflow_version,
    safe_metric("silhouette_score"),    # auc_roc slot → silhouette
    safe_metric("davies_bouldin_score"),# f1_score slot → davies-bouldin
    safe_metric("inertia"),             # precision slot → inertia
    safe_metric("cluster_size_cv"),     # recall slot → size CV
    None,                               # cv_auc_mean — not applicable
    None,                               # cv_auc_std  — not applicable
    None,                               # best_threshold — not applicable
    "cluster_id",
    "segment_name",
    json.dumps(encoded_feature_cols),
    json.dumps(primary_keys),
    str(job_id),
    str(parent_job),
    str(partition),
    "production",
    now, now, None,
    "sahil.prusty09@gmail.com",
)]

registry_df = spark.createDataFrame(registry_row, schema=registry_schema)

DeltaTable.forName(spark, registry_table) \
    .alias("t") \
    .merge(registry_df.alias("s"), "t.model_name = s.model_name") \
    .whenMatchedUpdate(set={
        "run_id"         : "s.run_id",
        "model_uri"      : "s.model_uri",
        "mlflow_version" : "s.mlflow_version",
        "auc_roc"        : "s.auc_roc",
        "f1_score"       : "s.f1_score",
        "precision_score": "s.precision_score",
        "recall_score"   : "s.recall_score",
        "feature_cols"   : "s.feature_cols",
        "status"         : "s.status",
        "updated_at"     : "s.updated_at",
        "job_id"         : "s.job_id",
        "partition"      : "s.partition",
    }) \
    .whenNotMatchedInsertAll() \
    .execute()

print(f"✅ Model registry updated : {registry_table}")
print(f"\n📋 Registry entry:")
display(spark.table(registry_table).filter(f"model_name = '{model_name_reg}'"))

In [0]:
spark.sql(f"OPTIMIZE {target_table} ZORDER BY (customer_unique_id, cluster_id)")
print(f"✅ OPTIMIZE complete : {target_table}")

print(f"\n{'='*60}")
print(f"  🎉 CUSTOMER SEGMENTATION COMPLETE")
print(f"{'='*60}")
print(f"  Customers segmented : {len(df_output):,}")
print(f"  Segments created    : {k_used}")
print(f"  Silhouette score    : {silhouette:.4f}")
print(f"  Target table        : {target_table}")
print(f"  Job ID              : {job_id}")
print(f"{'='*60}")